In [ ]:
# !pip install svgpathtools
# !pip install cairosvg

In [ ]:
# Read SVG into a list of path objects and list of dictionaries of attributes 
from svgpathtools import svg2paths, wsvg
paths, attributes = svg2paths('pdftest_p0.svg')

# Update: You can now also extract the svg-attributes by setting
# return_svg_attributes=True, or with the convenience function svg2paths2
from svgpathtools import svg2paths2
paths, attributes, svg_attributes = svg2paths2('pdftest_p0.svg')

#Print shapes of paths, attributes, svg_attributes
print(f'SVG attributes are: {svg_attributes}')
print(f' There are {len(paths)} paths in the file')
print(f'There are {len(attributes)} attributes in the file')

In [ ]:
#Iterate through the paths. Calculate the length of each path only along y-axis (end_y - start_y)
import matplotlib.pyplot as plt

path_lengths_y = []
for path in paths:
    path_lengths_y.append(abs(path.start.imag - path.end.imag))

plt.hist(path_lengths_y, bins=50)
plt.xlabel('Path Length along y-axis')

average_path_length_y = sum(path_lengths_y) / len(path_lengths_y)
print(f'The average path length along y-axis is {average_path_length_y}')

In [ ]:
# Let's make a new SVG that's identical to the first
wsvg(paths, attributes=attributes, svg_attributes=svg_attributes, filename='output1.svg')
#Display the output1.svg file
from IPython.display import SVG
SVG(filename='output1.svg')

In [ ]:
#Lets create new paths and attributes lists, with only some path range to understand the sapling rate
starting_path = 0#350
end_path =5 #450
new_paths = paths[starting_path:end_path]
new_attributes = attributes[starting_path:end_path]

wsvg(new_paths, attributes=new_attributes, svg_attributes=svg_attributes, filename='output1.svg')
#Display the output1.svg file
from IPython.display import SVG
SVG(filename='output1.svg')

In [ ]:
#Print the first 200 characters of the SVG file
with open('pdftest_p0.svg', 'r') as file:
    data = file.read().replace('\n', '')

chunks = 121
for i in range(0, len(data), chunks):
    print(data[i:i+chunks])

Primero puedo encontrar con la senhal de calibracion, donde es el 0 en magnitud, donde es el 1 en magnitud, y el rango en coordenadas que corresponde al periodo T, donde T es el ancho en el tiempo de la senhal de calibracion

Puedo encontrar donde comienza y termina la senhal de calibracion comprobando que es una linea super recta. Tratare de iterar por los primeros paths y encontrar los extremos de la senhal de calibracion

In [ ]:
#Iterate throufhg the first TEST paths and attributes
TEST = 100

# #Now, we iterate through the paths, recording when Y starts and ends being the same (straight line)
for j in range(0,len(paths)-1):
    #Get the length of the path. The calibration signal is a big jump
    length = paths[j].length()
    if length > 1:
        #Save the current X as x-start-calibration-signal
        x_start_first_line = paths[j].start.real
        x_end_first_line = paths[j].end.real
        y_start_first_line = paths[j].start.imag
        y_end_first_line = paths[j].end.imag
        print(f"Index before first line: {j}")
        length_first_line = length
        j+=1
        break

#Now we continue iterating unti we fint the second big jump, which denotes the third line
for j in range (j,len(paths)-1):
    length = paths[j].length()
    if length > 1:
        #Save the current X as x-start-calibration-signal
        x_start_second_line = paths[j].start.real
        x_end_second_line = paths[j].end.real
        y_start_second_line = paths[j].start.imag
        y_end_second_line = paths[j].end.imag
        length_third_line = length
        print(f"Index before second line: {j}")
        break
length_second_line = y_end_first_line - y_start_second_line
print(f"Length of second line: {length_second_line}")

print(f"First line starts at {x_start_first_line}, {y_start_first_line} and ends at {x_end_first_line},{y_end_first_line}")
print(f"Second line starts at {x_end_first_line}, {y_end_first_line} and ends at {x_start_second_line}, {y_start_second_line}")
print(f"Third line starts at {x_start_second_line}, {y_start_second_line} and ends at {x_end_second_line}, {y_end_second_line}")
print(f"Length of first line is {length_first_line}, second line is {length_second_line} and third line is {length_third_line}")
#Calibration signals are 1 mv high (10mm/mV and 10mm high ) , and 5mm (0.2s) wide (25mm/sec)
#Now we just calculate how much every unit is in mv and seconds
resolution_mv = 1/length_first_line
resolution_s = 0.2/length_second_line
print(f"Every unit in the SVG in mv is {resolution_mv} and in seconds is {resolution_s}= {1/resolution_s} Hz")
print(f"Step size for time axis is {resolution_s*average_path_length_y} s - {1/(resolution_s*average_path_length_y)} Hz")

In [ ]:
#Let's check for continuity intervals in the signal, to make sure I undesrtand how are they being drawn
intervals_start = []
intervals_end = []

#iterate through all paths
intervals_start.append(0)
for j in range(0,len(paths)-1):

#Continuity is when path x[j].end.real and path x[j+1].start.real are the same
    if paths[j].end.real == paths[j+1].start.real:
        pass #continuity
    else:
        print(f"Discontinuity between {j} and {j+1}")
        intervals_end.append(j)
        intervals_start.append(j+1)

intervals_end.append(len(paths)-1)
print(f'Found {len(intervals_start)} intervals')
print(intervals_start)
print(intervals_end)


In [ ]:
#Lets create new paths and attributes lists, with only some path range to understand the sapling rate
for i in range (len(intervals_start)):

    starting_path = intervals_start[i]
    end_path = intervals_end[i]
    new_paths = paths[starting_path:end_path]
    new_attributes = attributes[starting_path:end_path]

    wsvg(new_paths, attributes=new_attributes, svg_attributes=svg_attributes, filename=f'output{i}.svg')

In [ ]:
#Display the output1.svg file
from IPython.display import SVG
SVG(filename='output20.svg')

#QAfter checking, lines 0 to 2 are calibration, 3 to 14 the first leads
#Then 15 to 17 the calibration, then 18 to 20 the 10 second leads

In [ ]:
#Lets retrieve the paths for the first calibration signal

#Create a function that accepts the calibration signals and returns the X coordinate at origin and at 1
def get_calibration_coordinates(paths_calibration, debug = False):
    #Iterate through the paths, until we find a path with length > 1
    for j in range(0,len(paths_calibration)-1):
        #Get the length of the path. The calibration signal is a big jump
        length = paths_calibration[j].length()
        if length > 1:
            #Save the current X as x-start-calibration-signal
            x_origin = paths_calibration[j].start.real
            x_unit = paths_calibration[j].end.real
            y_start_first_line = paths_calibration[j].start.imag
            y_end_first_line = paths_calibration[j].end.imag
            length_first_line = length
            if (debug):
                print(f"Length of path {j} is {length_first_line}. X axis (magnitude)")
                print(f"Resolution of magnitude is {1/length_first_line}")
                print(f"X coordinate at origin: {x_origin}")
                print(f"X coordinate at unit magnitude: {x_unit}")
            j+=1
            break

        #Get the second encountered big lenght, which belongs to the second line, y-axis
    for j in range (j,len(paths_calibration)-1):
        length = paths_calibration[j].length()
        if length > 1:
            y_start_second_line = paths_calibration[j].start.imag
            y_end_second_line = paths_calibration[j].end.imag
            if (debug):
                print(f"Length of path {j} is {length_first_line}. X axis (magnitude) third line")
            break
    y_length = y_end_first_line - y_start_second_line 
    print(f"Length of second line y-axis (time): {y_length}")
    return x_origin, x_unit, y_length

x_origin_1, x_unit_1, y_length_1 = get_calibration_coordinates(paths[intervals_start[0]:intervals_end[0]], True) #First calibration signal
x_origin_2, x_unit_2, y_length_2 = get_calibration_coordinates(paths[intervals_start[1]:intervals_end[1]], True) #second calibration signal
x_origin_3, x_unit_3, y_length_3 = get_calibration_coordinates(paths[intervals_start[2]:intervals_end[2]], True) #third calibration signal
x_origin_4, x_unit_4, y_length_4 = get_calibration_coordinates(paths[intervals_start[15]:intervals_end[15]], True) #fourth calibration signal
x_origin_5, x_unit_5, y_length_5 = get_calibration_coordinates(paths[intervals_start[16]:intervals_end[16]], True) #fifth calibration signal
x_origin_6, x_unit_6, y_length_6 = get_calibration_coordinates(paths[intervals_start[17]:intervals_end[17]], True) #sixth calibration signal

In [ ]:
#CHECKING IF THE LOGIC IS RIGHT

#Now lets get the paths for the first lead
paths_lead_1 = paths[intervals_start[3]:intervals_end[3]]
#print the first 5 paths
for i in range(0,15):
    print(paths_lead_1[i])
    print(paths_lead_1[i].start.imag - paths_lead_1[i].end.imag)

##ASIDE. I find that not every path has the same time axis increase, there is a slight deviation. Let's understand why

#Find common denominator for 0.14178466796875 and 0.1417236328125
from fractions import Fraction
print(Fraction(0.14178466796875).limit_denominator())
print(Fraction(0.1417236328125).limit_denominator())
print(f'if we multiply the 2nd one, num and den by 2 we get {Fraction(0.1417236328125*2).limit_denominator().numerator*2} and {Fraction(0.1417236328125*2).limit_denominator().denominator*2}')
#We can see that they are almost the same
print(f'We can use (2322.5*2)/(2*16384) = {2322.5*2}/{2*16384}')
print(f'-------------------------------------------------')
print(f"HENCE, TIME STEP IS 4645/32768")
print(f'-------------------------------------------------')

In [ ]:
#Now get sampling rates (for each in case there is somehow an error)
#Calibration signals are 1 mv high (10mm/mV and 10mm high ) , and 5mm (0.2s) wide (25mm/sec)

time_step = 4645/32768  #According to the justification in the previous cell

#To get the time_step in seconds:
time_step1_s = (0.2*time_step)/y_length_1 
time_step2_s = (0.2*time_step)/y_length_2
time_step3_s = (0.2*time_step)/y_length_3
time_step4_s = (0.2*time_step)/y_length_4
time_step5_s = (0.2*time_step)/y_length_5
time_step6_s = (0.2*time_step)/y_length_6
print(f"Time step 1 is {time_step1_s} s. Frequency is {1/time_step1_s} Hz")
print(f"Time step 2 is {time_step2_s} s. Frequency is {1/time_step2_s} Hz")
print(f"Time step 3 is {time_step3_s} s. Frequency is {1/time_step3_s} Hz")
print(f"Time step 4 is {time_step4_s} s. Frequency is {1/time_step4_s} Hz")
print(f"Time step 5 is {time_step5_s} s. Frequency is {1/time_step5_s} Hz")
print(f"Time step 6 is {time_step6_s} s. Frequency is {1/time_step6_s} Hz")


In [ ]:
#Iterate throught the paths for the first lead, calculating the magnitude axis (y, imag)
#TESTING, THEN I WILL FACTOR INTO A FUNCTION

#I can substract the x-origin and then build a numpy array with the values
import numpy as np
lead_1 = np.zeros(len(paths_lead_1))

for i, path in enumerate (paths_lead_1):
#    lead_1[i] = path.start.real - x_origin_1   #It was inverted I think
    lead_1[i] = x_origin_1 - path.start.real  #Trying it like this

#Now we can plot the lead
plt.figure(figsize=(20,5))
plt.plot(lead_1)

#Define number of bits for each mv 
#I read that full range of 5 mv can be considered 
#If I use float, then I can just convert the magnitude to mv

# I would just scale all values by a factor
#x_origin_1 - x_unit_1 #This is 1 mv according to the standard calibration signal
lead_1_mv = lead_1 * 1/(x_origin_1 - x_unit_1)
plt.figure(figsize=(20,5))
plt.plot(lead_1_mv)

In [ ]:
#Define function to get the lead

def get_lead_signal (paths_lead, x_origin, x_unit, original_unit_or_mv = 'original', plot_signal = False):
    lead_signal = np.zeros(len(paths_lead))

    for i, path in enumerate (paths_lead):
        #lead_signal[i] = path.start.real - x_origin  #It was inverted I think
        lead_signal[i] = x_origin - path.start.real  #Trying it like this

    if original_unit_or_mv == 'mv':
        # I would just scale all values by a factor
        #x_origin_1 - x_unit_1 #This is 1 mv according to the standard calibration signal
        lead_signal = lead_signal * 1/(x_origin_1 - x_unit_1)
    if (plot_signal):
        plt.figure(figsize=(20,5))
        plt.plot(lead_signal)
    return lead_signal

def get_all_lead_signals (paths, intervals_start, intervals_end, 
                          x_origin_1, x_origin_2, x_origin_3, 
                          x_unit_1, x_unit_2, x_unit3, 
                          original_unit_or_mv = 'original', plot_signal = False):
    #TODO refactor this
    paths_lead_1 = paths[intervals_start[3]:intervals_end[3]]
    paths_lead_2 = paths[intervals_start[4]:intervals_end[4]]
    paths_lead_3 = paths[intervals_start[5]:intervals_end[5]]
    paths_lead_4 = paths[intervals_start[6]:intervals_end[6]]
    paths_lead_5 = paths[intervals_start[7]:intervals_end[7]]
    paths_lead_6 = paths[intervals_start[8]:intervals_end[8]]
    paths_lead_7 = paths[intervals_start[9]:intervals_end[9]]
    paths_lead_8 = paths[intervals_start[10]:intervals_end[10]]
    paths_lead_9 = paths[intervals_start[11]:intervals_end[11]]
    paths_lead_10 = paths[intervals_start[12]:intervals_end[12]]
    paths_lead_11 = paths[intervals_start[13]:intervals_end[13]]
    paths_lead_12 = paths[intervals_start[14]:intervals_end[14]]
    
    #Create the array that will store all leads
    ECG = np.zeros((len(lead_1),12))

    #Get the lead signals, mapping them to the appropriate origin
    ECG[:,0] = get_lead_signal(paths_lead_1, x_origin_1, x_unit_1, original_unit_or_mv, plot_signal)
    ECG[:,1] = get_lead_signal(paths_lead_2, x_origin_2, x_unit_2, original_unit_or_mv, plot_signal)
    ECG[:,2] = get_lead_signal(paths_lead_3, x_origin_3, x_unit_3, original_unit_or_mv, plot_signal)
    ECG[:,3] = get_lead_signal(paths_lead_4, x_origin_1, x_unit_1, original_unit_or_mv, plot_signal)
    ECG[:,4] = get_lead_signal(paths_lead_5, x_origin_2, x_unit_2, original_unit_or_mv, plot_signal)
    ECG[:,5] = get_lead_signal(paths_lead_6, x_origin_3, x_unit_3, original_unit_or_mv, plot_signal)
    ECG[:,6] = get_lead_signal(paths_lead_7, x_origin_1, x_unit_1, original_unit_or_mv, plot_signal)
    ECG[:,7] = get_lead_signal(paths_lead_8, x_origin_2, x_unit_2, original_unit_or_mv, plot_signal)
    ECG[:,8] = get_lead_signal(paths_lead_9, x_origin_3, x_unit_3, original_unit_or_mv, plot_signal)
    ECG[:,9] = get_lead_signal(paths_lead_10, x_origin_1, x_unit_1, original_unit_or_mv, plot_signal)
    ECG[:,10] = get_lead_signal(paths_lead_11, x_origin_2, x_unit_2, original_unit_or_mv, plot_signal)
    ECG[:,11] = get_lead_signal(paths_lead_12, x_origin_3, x_unit_3, original_unit_or_mv, plot_signal)
    
    return ECG


In [ ]:
ECG_12_lead_mV = get_all_lead_signals(paths, intervals_start, intervals_end,
                                      x_origin_1, x_origin_2, x_origin_3,
                                      x_unit_1, x_unit_2, x_unit_3,
                                      'mv', True)

ECG_12_lead_raw_mag = get_all_lead_signals(paths, intervals_start, intervals_end,
                                      x_origin_1, x_origin_2, x_origin_3,
                                      x_unit_1, x_unit_2, x_unit_3,
                                      'original', True)                                      

In [ ]:
#Save th ECG signal to a file, to test denoising in another notebook
#np.save('ECG_12_lead.npy', ECG_12_lead)

In [ ]:
#HELPER FUNCTIONS (COPIED FROM MY OTHER NOTEBOOK)
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

def plot_ecg_frequency_spectrum(ECG_sample, fs=500, x_max=None, interactive=False):
    """
    Plots the frequency spectrum of an ECG signal.

    Parameters:
    ECG_sample (numpy array): The ECG signal sample.
    fs (int): The sample rate in Hz. Default is 500 Hz.
    x_max (float): The upper limit for the x-axis (frequency). Default is None.
    """
    # Perform FFT on the ECG sample
    ecg_fft = np.fft.fft(ECG_sample)
    freq = np.fft.fftfreq(len(ecg_fft), d=1/fs)  # Frequency axis
    mag_spectrum = np.abs(ecg_fft)

    # Filter to show only the positive frequencies
    positive_freq_indices = freq >= 0
    freq = freq[positive_freq_indices]
    mag_spectrum = mag_spectrum[positive_freq_indices]

    # If x_max is specified, filter the frequencies and magnitudes
    if x_max is not None:
        freq = freq[freq <= x_max]
        mag_spectrum = mag_spectrum[:len(freq)]

    if interactive: #We will plot with plotyly express

        # Create a DataFrame for Plotly Express
        data = {"Frequency": freq, "Magnitude": mag_spectrum}
        df = pd.DataFrame(data)

        # Plot the interactive frequency spectrum
        fig = px.line(df, x="Frequency", y="Magnitude", labels={"Frequency": "Frequency (Hz)", "Magnitude": "Magnitude"})
        fig.update_layout(
            title="Frequency Spectrum of ECG Signal",
            xaxis_title="Frequency (Hz)",
            yaxis_title="Magnitude",
            hovermode="x unified",
        )

        # Set the x-axis range if x_max is specified
        if x_max is not None:
            fig.update_xaxes(range=[0, x_max])

        fig.show()
    else:
        # Plot the frequency spectrum
        plt.figure(figsize=(10, 6))
        plt.plot(freq, mag_spectrum)
        plt.title("Frequency Spectrum of ECG Signal")
        plt.xlabel("Frequency (Hz)")
        plt.ylabel("Magnitude")
        plt.grid(True)
        plt.show()
    #return the spectrum
    return freq, mag_spectrum


def plot_ECG (random_ECG, random_patient_id, random_ecg_time, random_time_since_procedure):
    # Plot the ECG, show each channel in one subplot
    fig, axs = plt.subplots(random_ECG.shape[1], 1, figsize=(15, 12), sharex=True)
    for i in range(random_ECG.shape[1]):
        axs[i].plot(random_ECG[:, i])
        axs[i].set_title(f'Channel {i+1}')
    fig.suptitle(f'ECG of patient {random_patient_id} at time {random_ecg_time}. Time since procedure: {random_time_since_procedure} days')
    plt.show()


In [ ]:
#Plot the frequency spectrum of the first lead
plot_ecg_frequency_spectrum(ECG_12_lead_raw_mag[:,0], fs=500, interactive=True)

**LEts denoise this sample ECG**

In [ ]:
import numpy as np

# Load the numpy file
ECG_methodist_filtered_EMD = np.load('ECG_12_lead_filtered_EMD.npy')

# Convert back to raw magnitude
ECG_methodist_filtered_EMD = (x_origin_1 - x_unit_1) * ECG_methodist_filtered_EMD

# Now iterate through the paths, and replace the x start values with the denoised ones.
# Also replace the x end values with the next x start value.
# This way, we can create a new path, with the same attributes, but with the denoised values.
paths_denoised = paths.copy()
for lead in range(12):
    interval_idx_for_first_lead = 3
    paths_for_this_lead = paths_denoised[intervals_start[interval_idx_for_first_lead+lead]:intervals_end[interval_idx_for_first_lead+lead]]

    for i, path in enumerate(paths_for_this_lead):

        # Every 4 leads have different x_origin_1, x_origin_2, x_origin_3
        if (lead+1) % 3 == 1:  # +1 to avoid division by zero
            appropriate_x_origin = x_origin_1
        elif (lead+1) % 3 == 2:
            appropriate_x_origin = x_origin_2
        else:
            appropriate_x_origin = x_origin_3

        path.start = complex(ECG_methodist_filtered_EMD[i,lead] + appropriate_x_origin, path.start.imag) # Shifting back to the appropriate origin coordinate
 
        if i < len(paths_for_this_lead) - 1:
            path.end = complex(ECG_methodist_filtered_EMD[i + 1,lead] + appropriate_x_origin, path.end.imag) #Shifting the end of each path
        else:
            path.end = complex(ECG_methodist_filtered_EMD[i,lead] + appropriate_x_origin, path.end.imag)

    print(f"Length of paths_for_this_lead is {len(paths_for_this_lead)}")
    print(f'path_lead_1[0] is {paths_for_this_lead[0]}')
    paths_denoised[intervals_start[interval_idx_for_first_lead+lead]:intervals_end[interval_idx_for_first_lead+lead]] = paths_for_this_lead

#paths_denoised = paths[intervals_start[interval_idx_for_first_lead]:intervals_end[interval_idx_for_first_lead]]


In [ ]:
#Create an svg concatenating the paths
wsvg(paths_denoised, attributes=attributes, svg_attributes=svg_attributes, filename='output_denoised.svg')
#Display the output1.svg file
from IPython.display import SVG
SVG(filename='output_denoised.svg')

In [ ]:
paths_denoised = paths.copy()
#paths[intervals_start[3]:intervals_end[3]] = paths_lead_1
#THen create a svg with paths and plot it
wsvg(paths, attributes=attributes, svg_attributes=svg_attributes, filename='output_denoised.svg')
#Display the output1.svg file
from IPython.display import SVG
SVG(filename='output_denoised.svg')

In [ ]:
#Examine the paths for the first lead
for i in range(0,15):
    print(paths_lead_1[i])
    print(paths_lead_1[i].start.imag - paths_lead_1[i].end.imag)

In [ ]:
# import cairosvg

# # Specify the input and output file paths
# input_svg = 'pdftest_p0.svg'
# output_png = 'cairo_pdftest_p0.png'

# # Convert SVG to PNG
# cairosvg.svg2png(url=input_svg, write_to=output_png, output_width=8000, output_height=8000)

#I dont think we are going this route... we lose a lot of information and with the noisy ecgs will get just black spots